# Simulation Iteration Tracker

시뮬레이션 DB(`data/simulation.db`)에서 RL run/iteration별 감정/관계/지식 변화를 확인합니다.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "data").exists() and (p / "config").exists():
        ROOT = p
        break
DB = ROOT / "data" / "simulation.db"
con = sqlite3.connect(DB)

def q(sql, params=None):
    return pd.read_sql_query(sql, con, params=params or {})

DB


In [ ]:
# episode run instances (new tracking table)
episode_runs = q("""
SELECT id AS episode_run_id, episode_id, run_id, iteration, phase, source, status, started_at, ended_at
FROM episode_runs
ORDER BY started_at DESC
""")
episode_runs.head(30)


In [ ]:
# Emotions tagged by iteration
emotions = q("""
SELECT episode_id, run_id, iteration, phase, episode_run_id, agent_id, turn, emotion_type, intensity, timestamp
FROM emotions
WHERE run_id IS NOT NULL
ORDER BY iteration, episode_id, agent_id, turn
""")
emotions.head(20)


In [ ]:
# Example: protagonist emotion trajectory across iterations for a chosen episode/emotion
TARGET_EPISODE = "ep08_personal_risk"
TARGET_AGENT = "kim_sumin"
TARGET_EMOTION = "fear"

plot_df = emotions[(emotions.episode_id == TARGET_EPISODE) & (emotions.agent_id == TARGET_AGENT) & (emotions.emotion_type == TARGET_EMOTION)].copy()
plot_df


In [ ]:
if not plot_df.empty:
    plt.figure(figsize=(10,4))
    if sns is not None:
        sns.lineplot(data=plot_df, x="turn", y="intensity", hue="iteration", marker="o")
    else:
        for it, g in plot_df.groupby("iteration"):
            plt.plot(g["turn"], g["intensity"], marker='o', label=f'iter {it}')
        plt.legend()
    plt.title(f"{TARGET_EPISODE} | {TARGET_AGENT} | emotion={TARGET_EMOTION}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Relationship values by iteration (latest turn per episode/iteration)
rels = q("""
SELECT run_id, iteration, episode_id, agent1_id, agent2_id, value, turn, updated_at
FROM relationships
WHERE run_id IS NOT NULL
ORDER BY iteration, episode_id, agent1_id, agent2_id, turn
""")
rels.head(20)


In [ ]:
if not rels.empty:
    latest_rels = (rels.sort_values(["run_id","iteration","episode_id","agent1_id","agent2_id","turn"])
                     .groupby(["run_id","iteration","episode_id","agent1_id","agent2_id"], as_index=False)
                     .tail(1))
    latest_rels.head(20)


In [ ]:
# Clue discovery counts by iteration/episode
clues = q("""
SELECT run_id, iteration, episode_id, agent_id, clue_id, discovered_turn
FROM agent_knowledge
WHERE run_id IS NOT NULL
""")
if not clues.empty:
    clue_counts = clues.groupby(["run_id","iteration","episode_id"], as_index=False).size().rename(columns={"size":"clues_discovered"})
    display(clue_counts.head(30))


## Notes
- `run_id`는 RL trainer 실행 단위, `iteration`은 RL 라운드입니다.
- `episode_run_id`로 같은 에피소드의 반복 실행을 정확히 구분할 수 있습니다.
- 이전 DB(트래킹 컬럼 없는 상태) 데이터는 `run_id IS NULL`로 남을 수 있습니다.